# Ngày 28 · Dự án: CNN với nhiều seed

- Cố định mọi nguồn ngẫu nhiên để một lần chạy lặp lại được
- Báo cáo kết quả bằng trung bình ± độ lệch chuẩn và khoảng tin cậy qua nhiều seed
- So hai cấu hình bằng hiệu cặp (paired) trên cùng các seed

**Đọc:**
- Bouthillier et al. (2021), Accounting for Variance in Machine Learning Benchmarks, Proceedings of MLSys 3
- Tài liệu PyTorch, Reproducibility (torch.manual_seed, torch.use_deterministic_algorithms)
- Henderson et al. (2018), Deep Reinforcement Learning that Matters, AAAI (phần về seed)

In [ ]:
import json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd

DATA = next((p for p in (Path('data'), Path('../data')) if p.is_dir()), Path('data'))

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
try:
    import torch
    from torch import nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError('Kernel này chưa có PyTorch. Cài bằng Python của Jupyter: python -m pip install -r requirements-py30.txt') from error
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
# load_digits (UCI Optical Recognition of Handwritten Digits): 1797 ảnh 8×8, chia 80/20 phân tầng, seed 0.
_Xd, _yd = load_digits(return_X_y=True)
_itr, _iva = train_test_split(np.arange(len(_yd)), test_size=0.2, random_state=0, stratify=_yd)
X_tr = torch.tensor(_Xd[_itr] / 16, dtype=torch.float32); y_tr = torch.tensor(_yd[_itr])
X_va = torch.tensor(_Xd[_iva] / 16, dtype=torch.float32); y_va = torch.tensor(_yd[_iva])
def _loader(X, y, batch_size, seed):
    return torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=batch_size, shuffle=True,
                                       generator=torch.Generator().manual_seed(seed))
def _train_epoch(model, loader, loss_fn, optimizer):
    model.train(); total, n = 0.0, 0
    for xb, yb in loader:
        optimizer.zero_grad(); loss = loss_fn(model(xb), yb); loss.backward(); optimizer.step()
        total += loss.item() * len(yb); n += len(yb)
    return total / n
def _accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return float((model(X).argmax(1) == y).float().mean())
from scipy import stats
class _CNN28(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(64, 10))
    def forward(self, x): return self.f(x)
def train_digits_cnn(epochs=5):
    """Huấn luyện CNN nhỏ trên load_digits KHÔNG tự đặt seed: kết quả phụ thuộc trạng thái ngẫu nhiên hiện tại."""
    m = _CNN28(); opt = torch.optim.Adam(m.parameters(), 1e-2)
    L = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X_tr.view(-1, 1, 8, 8), y_tr), batch_size=64, shuffle=True)
    for _ in range(epochs): _train_epoch(m, L, nn.CrossEntropyLoss(), opt)
    return _accuracy(m, X_va.view(-1, 1, 8, 8), y_va)


## Seed, tổng hợp nhiều lần chạy và so sánh cặp

Gợi ý: seed_everything: random.seed, np.random.seed, torch.manual_seed. summarize: mean, std với ddof = 1, half = stats.t.ppf(0.975, n − 1)·std/√n. paired_compare: d = a − b rồi summarize(d).

In [ ]:
def seed_everything(seed):
    raise NotImplementedError

def run_seeds(train_once, seeds):
    # Gọi seed_everything(s) rồi train_once() cho từng seed; trả np.array điểm.
    raise NotImplementedError

def summarize(scores):
    # Trả dict: mean, std (ddof=1), ci_low, ci_high (khoảng tin cậy 95% theo phân phối t).
    raise NotImplementedError

def paired_compare(a, b):
    # a, b: điểm của hai cấu hình trên cùng danh sách seed. Trả summarize(a − b).
    raise NotImplementedError


In [ ]:
_s28 = np.array([.972, .983, .981, .969, .969])
def _det():
    a = run_seeds(lambda: (random.random(), float(np.random.rand()), float(torch.rand(1))), [3, 3, 4])
    return np.allclose(a[0], a[1]) and not np.allclose(a[0], a[2])
def _cnn_det():
    r = run_seeds(lambda: train_digits_cnn(2), [7, 7])
    return r[0] == r[1]
_report('d28-seeds',
    _check('Cùng seed cho cùng số ngẫu nhiên ở random, NumPy và PyTorch', lambda: (_det(), 'Seed 3 hai lần giống nhau, seed 4 khác.')),
    _check('Huấn luyện CNN lặp lại được với cùng seed', lambda: (_cnn_det(), 'train_digits_cnn không tự đặt seed; run_seeds phải làm.')),
    _check('summarize đúng trung bình, độ lệch chuẩn mẫu và khoảng tin cậy t', lambda: ((lambda r: np.isclose(r['mean'], .9748) and np.isclose(r['std'], _s28.std(ddof=1)) and np.isclose(r['ci_high'] - r['mean'], stats.t.ppf(.975, 4) * _s28.std(ddof=1) / math.sqrt(5)))(summarize(_s28)), 'Năm điểm accuracy của CNN.')),
    _check('paired_compare dùng hiệu từng cặp', lambda: ((lambda r: np.isclose(r['mean'], .01) and np.isclose(r['std'], np.std([.01, .012, .008, .011, .009], ddof=1)))(paired_compare(_s28, _s28 - np.array([.01, .012, .008, .011, .009]))), 'Hiệu cặp ổn định thì khoảng tin cậy hẹp.')),
)


## Dự án

Trong notebook cloud trên GPU, huấn luyện CNN Fashion-MNIST với 5 seed cho hai cấu hình (ví dụ có và không có Dropout). Báo cáo bảng accuracy test từng seed, trung bình ± độ lệch chuẩn, khoảng tin cậy 95% của hiệu cặp, và kết luận cấu hình nào tốt hơn có chắc chắn không. Ghi rõ GPU có thể không tất định tuyệt đối.